# CodeT5-small: which output format works best? (A / B / C, 7 epochs each)

Trains **the same model on the same rows three times, once per output format**, then compares them on held-out code,
HumanEval and ClassEval, and writes conclusions computed from the results.

| Format | What the model is asked to write |
|---|---|
| **A** | the code exactly as it is in the dataset (control) |
| **B** | canonical code: docstrings and comments dropped, layout by `ast.unparse` |
| **C** | the full AST as a compact tree; a deterministic converter renders it back to Python |

Set `FORMATS = ["a", "b", "c", "d"]` in Cell 1 to also train **D** (statement tree, shorter than C).

## How to run
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload `dataset_formats.parquet` (from the dataset builder notebook) to `/content`.
3. Run all cells. Cell 8 is a preflight that checks everything that could fail *before* training starts.
4. Cell 10 prints the comparison and conclusions. Cell 11 downloads `results.json`, `report.txt` and every prediction.

Training is the long part. My estimate from earlier CodeT5-small runs is roughly 30-40 minutes per format on a T4, about
2 hours for three formats plus evaluation. That is an estimate, not a measurement for this notebook; set
`TRAIN_FRACTION = 0.5` in Cell 1 for about half. Each finished format is saved, so after a disconnect re-run Cells 1-8 and Cell 9
and finished formats are skipped.

## What makes the comparison fair
- **Same rows:** only rows whose target fits in 512 tokens in *every* format (and whose input fits) are used, so no format is
  trained or tested on easier data. Splits come from the Parquet and are per code unit (no leakage).
- **Same settings:** 7 epochs, learning rate 1e-4, batch 8x2, seed 42, best-validation-loss weights kept.
- **Judged on Python, not on text:** every output (trees included) is rendered to Python, then scored: valid, exact match against the
  format's own reference answer (also ignoring unused imports), structural similarity, and golden input/output cases.
  An output that cannot be scored counts as a failure; it is never dropped from the denominator.
- **Reference points:** the same table shows what you get by copying the input unchanged for repair/refactor rows.
- **Benchmarks are scored by running the real tests:** HumanEval (checked with `check(candidate)`) and ClassEval (its unittest classes are
  run and must pass), each program in its own process with a time limit. Only problems whose reference solution passes the same
  harness in your Colab are used, so a failure is the model's, not the setup's.
- **Conclusions come from paired tests** on the same rows, with 95% intervals, and the notebook says so when no format produced usable output.

## Limits
- One seed and one learning rate per format; a format that needs different settings is not given them.
- HumanEval and ClassEval are zero-shot for a 60M model trained on a few thousand snippets. Expect scores near zero for every format;
  the notebook reports that instead of ranking noise. ClassEval uses 10 problems (short skeletons that fit the encoder window).
- Golden I/O exists only for typed top-level functions (about 7% of rows), so functional results rest on a small number of functions.

In [ ]:
# CELL 0 — INSTALL
!pip install -q transformers==4.46.3 datasets

In [ ]:
# CELL 1 — IMPORTS + CONFIG
# The allocator setting must be set BEFORE torch is imported, or it has no effect.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import ast, copy, gc, hashlib, json, math, random, re, signal, subprocess, sys, tempfile, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=SyntaxWarning)       # scraped code has odd escape sequences
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------- what to run ----------------
PARQUET_PATH = "/content/dataset_formats.parquet"   # upload the file made by the dataset notebook
WORK = Path("/content/format_exp")
BASE_MODEL = "Salesforce/codet5-small"              # 60M parameters
FORMATS = ["a", "b", "c"]                           # a=as-is code, b=canonical code, c=full compact tree
                                                    # add "d" (statement tree) to train it as well
FORMAT_NAMES = {"a": "A original", "b": "B canonical", "c": "C full tree", "d": "D stmt tree"}
TGT_COL = {"a": "tgt_a_code", "b": "tgt_b_canonical", "c": "tgt_c_tree", "d": "tgt_d_stmt_tree"}

# ---------------- training (identical for every format) ----------------
EPOCHS = 7                  # fixed; no early stopping
LR = 1e-4
BATCH_SIZE = 8
GRAD_ACCUM = 2              # effective batch 16
WARMUP_FRAC = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
TRAIN_FRACTION = 1.0        # 0.5 trains on half of the training units (about half the time)

# ---------------- evaluation ----------------
MAX_SRC = 512               # encoder window
MAX_NEW_TOKENS = 512        # output budget
GEN_BATCH = 16
EVAL_OWN_N = 300            # rows of the held-out test split to generate for
HE_N = 64                   # HumanEval problems (same selection rule as the earlier runs)
CLASSEVAL_N = 10            # ClassEval problems
CLASSEVAL_MAX_PROMPT_TOKENS = 450   # skeletons longer than this would be cut off by the encoder window

ALLOW_CPU = False           # training on CPU is far too slow; leave False in Colab

WORK.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = WORK / "results.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import transformers
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | pandas {pd.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()} ({torch.cuda.mem_get_info()[1]/1e9:.1f} GB)")
elif not ALLOW_CPU:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then run again.")
print(f"Formats: {[FORMAT_NAMES[f] for f in FORMATS]} | epochs {EPOCHS} | lr {LR} | batch {BATCH_SIZE}x{GRAD_ACCUM}")

In [ ]:
# CELL 2 — LOAD THE PARQUET AND CHOOSE THE ROWS EVERY FORMAT CAN BE TRAINED ON
# A format can only be compared fairly on rows where ALL formats fit in the 512-token output
# budget (and decode back to Python), otherwise each format would train and be tested on
# different data. So we keep the rows that fit for every format being trained.
pq_path = Path(PARQUET_PATH)
if not pq_path.exists():
    raise FileNotFoundError(f"Upload the Parquet from the dataset notebook to {PARQUET_PATH} first.")
df = pd.read_parquet(pq_path)
need = ["id", "unit_id", "split", "task", "domain", "unit_type", "input_text", "instruction", "context_code",
        "docstrings_kept", "has_golden_io", "golden_func", "golden_io_json", "fits512_input", "imports_pruned"] \
       + [TGT_COL[f] for f in FORMATS] + [f"fits512_{f}" for f in FORMATS] + [f"tok_{f}" for f in FORMATS] \
       + [f"ok_{f}" for f in FORMATS if f != "a"]
missing = [c for c in need if c not in df.columns]
assert not missing, f"the Parquet is missing columns: {missing}"
print(f"Loaded {len(df)} rows, {df.unit_id.nunique()} code units. imports_pruned = {bool(df.imports_pruned.any())}")

mask = df["fits512_input"].astype(bool)
for f in FORMATS:
    mask &= df[f"fits512_{f}"].astype(bool)
    if f != "a":
        mask &= df[f"ok_{f}"].astype(bool)
common = df[mask].copy()
print(f"Rows usable for every format: {len(common)} of {len(df)} ({len(common)/len(df)*100:.1f}%)")

def unit_fraction(u):
    return int(hashlib.sha1(("frac" + u).encode()).hexdigest()[:8], 16) % 1000 / 1000.0
train_df = common[common.split == "train"]
if TRAIN_FRACTION < 1.0:
    train_df = train_df[train_df.unit_id.map(unit_fraction) < TRAIN_FRACTION]
train_df = train_df.reset_index(drop=True)
val_df = common[common.split == "val"].reset_index(drop=True)
test_df = common[common.split == "test"].reset_index(drop=True)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}   (split is per code unit; no unit is in two splits)")
print("train rows by task:", train_df.task.value_counts().to_dict())
print("train rows by unit type:", train_df.unit_type.value_counts().to_dict())
print("dropped because some format did not fit 512 tokens, by unit type:")
dropped = df[~mask]
print("  ", dropped.unit_type.value_counts().to_dict())
print("mean target tokens on the rows used:")
for f in FORMATS:
    print(f"   {FORMAT_NAMES[f]:12s} {common[f'tok_{f}'].mean():6.0f}")
assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0, "an empty split; check the Parquet"

## Converters (the same deterministic code that built the Parquet) and the scoring kit

In [ ]:
# CELL 3 — FORMAT CONVERTERS: canonical code (B), compact tree (C), statement tree (D)
#
# All three are DETERMINISTIC and checked per row by a round-trip test:
#   format text -> rebuilt Python source -> same AST as the original (docstrings aside).
# No model is involved anywhere in this cell.
#
#   B  canonical code : docstrings/comments dropped, layout by ast.unparse
#   C  compact tree   : the FULL AST (every node, expressions included) as an S-expression
#   D  statement tree : statements as tree nodes, each expression kept as one text leaf
#
# C/D notation (single line, one space between items):
#   (TAG item ...)  node        [item ...]  list        None / True / False / ...   literals
#   10  1.5  -1     numbers     name  os.path  *   bare identifier-like strings
#   "text"  JSON string         x"ff00" bytes           j"2j" complex
#   `n + 1`         expression text leaf (D only; a literal backtick is written as two)
# Dropped from the text and restored on rebuild: ctx, type_comment, kind, type_ignores.
# type_params (generics) is written only when non-empty, as an optional trailing item.

import ast, copy, json, math, re, sys

sys.setrecursionlimit(20000)

TAGS = {
    "Module": "mod", "FunctionDef": "def", "AsyncFunctionDef": "adef", "ClassDef": "class",
    "Return": "ret", "Delete": "del", "Assign": "=", "AugAssign": "aug", "AnnAssign": "ann",
    "For": "for", "AsyncFor": "afor", "While": "while", "If": "if", "With": "with",
    "AsyncWith": "awith", "Raise": "raise", "Try": "try", "TryStar": "trystar",
    "Assert": "assert", "Import": "import", "ImportFrom": "from", "Global": "global",
    "Nonlocal": "nonlocal", "Expr": "expr", "Pass": "pass", "Break": "break",
    "Continue": "continue", "BoolOp": "bool", "NamedExpr": "walrus", "BinOp": "bin",
    "UnaryOp": "un", "Lambda": "lam", "IfExp": "ife", "Dict": "dict", "Set": "set",
    "ListComp": "lc", "SetComp": "sc", "DictComp": "dc", "GeneratorExp": "ge",
    "Await": "await", "Yield": "yield", "YieldFrom": "yieldfrom", "Compare": "cmp",
    "Call": "call", "FormattedValue": "fv", "JoinedStr": "fstr", "Constant": "c",
    "Attribute": "attr", "Subscript": "sub", "Starred": "star", "Name": "n", "List": "list",
    "Tuple": "tup", "Slice": "slice", "comprehension": "comp", "ExceptHandler": "exc",
    "arguments": "args", "arg": "arg", "keyword": "kw", "alias": "as", "withitem": "wi",
    "Add": "+", "Sub": "-", "Mult": "*", "MatMult": "@", "Div": "/", "Mod": "%", "Pow": "**",
    "LShift": "<<", "RShift": ">>", "BitOr": "|", "BitXor": "^", "BitAnd": "&",
    "FloorDiv": "//", "And": "and", "Or": "or", "Invert": "~", "Not": "not", "UAdd": "u+",
    "USub": "u-", "Eq": "==", "NotEq": "!=", "Lt": "<", "LtE": "<=", "Gt": ">", "GtE": ">=",
    "Is": "is", "IsNot": "isnot", "In": "in", "NotIn": "notin",
}
assert len(set(TAGS.values())) == len(TAGS), "duplicate short tag"
TAG_TO_CLS = {tag: getattr(ast, name) for name, tag in TAGS.items() if hasattr(ast, name)}

DROP_FIELDS = {"ctx", "type_comment", "kind", "type_ignores", "type_params"}

def kept_fields(cls):
    return [f for f in cls._fields if f not in DROP_FIELDS]

def _default_for(field):
    if field == "ctx":
        return ast.Load()
    if field in ("type_ignores", "type_params"):
        return []
    return None

def _jstr(s):
    out = json.dumps(s, ensure_ascii=False)
    try:
        out.encode("utf-8")
    except UnicodeEncodeError:
        out = json.dumps(s, ensure_ascii=True)
    return out

_IDENT = re.compile(r"[^\W\d]\w*(\.[^\W\d]\w*)*\Z")

def _is_bare(s):
    return s == "*" or (bool(_IDENT.match(s)) and s not in ("None", "True", "False"))

def _prim(v, quote_str):
    if v is None:
        return "None"
    if v is True:
        return "True"
    if v is False:
        return "False"
    if v is Ellipsis:
        return "..."
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        if math.isnan(v):
            raise ValueError("nan constant")
        if math.isinf(v):
            return "1e999" if v > 0 else "-1e999"
        return repr(v)
    if isinstance(v, complex):
        return "j" + _jstr(repr(v))
    if isinstance(v, bytes):
        return "x" + _jstr(v.hex())
    if isinstance(v, str):
        return _jstr(v) if (quote_str or not _is_bare(v)) else v
    raise TypeError(f"unsupported primitive {type(v)}")

def _ser(v, mode, in_const=False):
    if isinstance(v, ast.AST):
        if mode == "stmt" and isinstance(v, ast.expr):
            return "`" + ast.unparse(v).replace("`", "``") + "`"
        cls = type(v)
        parts = [TAGS.get(cls.__name__, cls.__name__)]
        is_const = isinstance(v, ast.Constant)
        for f in kept_fields(cls):
            parts.append(_ser(getattr(v, f, None), mode, is_const))
        if "type_params" in cls._fields and getattr(v, "type_params", None):
            parts.append(_ser(v.type_params, mode, False))   # generics: written only when present
        return "(" + " ".join(parts) + ")"
    if isinstance(v, list):
        return "[" + " ".join(_ser(x, mode, in_const) for x in v) + "]"
    return _prim(v, in_const)

def tree_to_text(tree, mode):
    assert mode in ("full", "stmt")
    return _ser(tree, mode)

# ---------- text -> tokens -> nested values -> AST -> source ----------
_STOP = set(' \t\r\n()[]"`')

def _read_jstr(s, i):
    j = i + 1
    while j < len(s):
        if s[j] == "\\":
            j += 2
            continue
        if s[j] == '"':
            return json.loads(s[i:j + 1]), j + 1
        j += 1
    raise ValueError("unterminated string")

def _lex(s):
    i, n, toks = 0, len(s), []
    while i < n:
        ch = s[i]
        if ch in " \t\r\n":
            i += 1
        elif ch in "()[]":
            toks.append(ch)
            i += 1
        elif ch == "`":
            i += 1
            buf = []
            while True:
                if i >= n:
                    raise ValueError("unterminated expression text")
                if s[i] == "`":
                    if i + 1 < n and s[i + 1] == "`":
                        buf.append("`")
                        i += 2
                        continue
                    i += 1
                    break
                buf.append(s[i])
                i += 1
            toks.append(("expr", "".join(buf)))
        elif ch == '"':
            val, i = _read_jstr(s, i)
            toks.append(("str", val))
        else:
            j = i
            while j < n and s[j] not in _STOP:
                j += 1
            word = s[i:j]
            if j < n and s[j] == '"' and word in ("x", "j"):
                val, j2 = _read_jstr(s, j)
                toks.append(("bytes", bytes.fromhex(val)) if word == "x" else ("complex", complex(val)))
                i = j2
            else:
                toks.append(("bare", word))
                i = j
    return toks

_INT = re.compile(r"-?\d+\Z")
_FLOAT = re.compile(r"-?(\d+\.\d*|\.\d+|\d+)([eE][+-]?\d+)?\Z")

def _bare_to_prim(w):
    if w == "None":
        return None
    if w == "True":
        return True
    if w == "False":
        return False
    if w == "...":
        return Ellipsis
    if _INT.match(w):
        return int(w)
    if _FLOAT.match(w):
        return float(w)
    return w

def _parse(toks, pos):
    if pos >= len(toks):
        raise ValueError("unexpected end of text")
    t = toks[pos]
    if t == "(":
        pos += 1
        if pos >= len(toks) or not (isinstance(toks[pos], tuple) and toks[pos][0] == "bare"):
            raise ValueError("node without a tag")
        tag = toks[pos][1]
        pos += 1
        args = []
        while pos < len(toks) and toks[pos] != ")":
            v, pos = _parse(toks, pos)
            args.append(v)
        if pos >= len(toks):
            raise ValueError("unclosed node")
        return ("node", tag, args), pos + 1
    if t == "[":
        pos += 1
        items = []
        while pos < len(toks) and toks[pos] != "]":
            v, pos = _parse(toks, pos)
            items.append(v)
        if pos >= len(toks):
            raise ValueError("unclosed list")
        return ("list", items), pos + 1
    if isinstance(t, tuple):
        kind, val = t
        if kind == "expr":
            return ("expr", val), pos + 1
        if kind == "bare":
            return ("prim", _bare_to_prim(val)), pos + 1
        return ("prim", val), pos + 1
    raise ValueError(f"unexpected token {t!r}")

def _parse_expr(text):
    try:
        return ast.parse("(" + text + "\n)", mode="eval").body
    except SyntaxError:
        # a lone starred item such as  *bases  is only legal inside a call
        call = ast.parse("_f(" + text + "\n)", mode="eval").body
        if len(call.args) == 1 and not call.keywords:
            return call.args[0]
        raise

def _cls_for(tag):
    cls = TAG_TO_CLS.get(tag)
    if cls is None:
        cls = getattr(ast, tag, None)
        if not (isinstance(cls, type) and issubclass(cls, ast.AST)):
            raise ValueError(f"unknown tag {tag!r}")
    return cls

def _build(v):
    kind = v[0]
    if kind == "list":
        return [_build(x) for x in v[1]]
    if kind == "prim":
        return v[1]
    if kind == "expr":
        return _parse_expr(v[1])
    _, tag, args = v
    cls = _cls_for(tag)
    fields = kept_fields(cls)
    has_tp = "type_params" in cls._fields and len(args) == len(fields) + 1
    if len(args) != len(fields) and not has_tp:
        raise ValueError(f"{tag}: expected {len(fields)} items, got {len(args)}")
    kwargs = {f: _build(a) for f, a in zip(fields, args)}
    if has_tp:
        kwargs["type_params"] = _build(args[-1])
    for f in cls._fields:
        if f not in kwargs:
            kwargs[f] = _default_for(f)
    return cls(**kwargs)

def text_to_source(text):
    """Deterministic render: tree text -> Python source (no model involved)."""
    toks = _lex(text)
    v, pos = _parse(toks, 0)
    if pos != len(toks):
        raise ValueError("trailing text after the tree")
    tree = ast.fix_missing_locations(_build(v))
    return ast.unparse(tree)

# ---------- docstring policy + canonical code + per-row checks ----------
def strip_docstrings(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.Module, ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
            b = node.body
            if b and isinstance(b[0], ast.Expr) and isinstance(b[0].value, ast.Constant) \
               and isinstance(b[0].value.value, str):
                node.body = b[1:] or [ast.Pass()]
    return tree

def norm_dump(x):
    tree = ast.parse(x) if isinstance(x, str) else copy.deepcopy(x)
    for n in ast.walk(tree):
        if isinstance(n, ast.Constant):
            n.kind = None
        if hasattr(n, "type_comment"):
            n.type_comment = None
    return ast.dump(tree)

def _check(name, fn):
    try:
        return fn(), None
    except Exception as e:
        return None, f"{name}: {type(e).__name__}: {str(e)[:120]}"

def used_names(tree):
    """Names the unit's own code refers to (excluding its import lines). Conservative: any word
    inside a string literal also counts, so imports used only in string annotations are kept."""
    used = set()
    for n in tree.body:
        if isinstance(n, (ast.Import, ast.ImportFrom)):
            continue
        for x in ast.walk(n):
            if isinstance(x, ast.Name):
                used.add(x.id)
            elif isinstance(x, ast.Constant) and isinstance(x.value, str):
                used.update(re.findall(r"[A-Za-z_]\w*", x.value))
    return used

def _bound_names(node):
    if isinstance(node, ast.Import):
        return [a.asname or a.name.split(".")[0] for a in node.names]
    return [a.asname or a.name for a in node.names if a.name != "*"]

def import_stats(tree, code):
    """(imports, unused imports, share of characters that are import lines) for a parsed unit."""
    used = used_names(tree)
    lines = code.split("\n")
    total = unused = chars = 0
    for n in tree.body:
        if isinstance(n, (ast.Import, ast.ImportFrom)):
            chars += sum(len(lines[i]) + 1 for i in range(n.lineno - 1, n.end_lineno))
            if isinstance(n, ast.ImportFrom) and n.module == "__future__":
                continue
            for b in _bound_names(n):
                total += 1
                unused += b not in used
    return total, unused, chars / max(1, len(code))

def prune_unused_imports(tree):
    """Drop imported names the rest of the unit never uses. Keeps __future__ and star imports;
    prunes nothing if the code uses globals()/eval()/exec()/vars()/__all__. Returns names removed."""
    used = used_names(tree)
    if used & {"globals", "locals", "eval", "exec", "vars", "__all__"}:
        return 0
    removed, new_body = 0, []
    for n in tree.body:
        if isinstance(n, ast.Import):
            keep = [a for a in n.names if (a.asname or a.name.split(".")[0]) in used]
        elif isinstance(n, ast.ImportFrom) and n.module != "__future__" and not any(a.name == "*" for a in n.names):
            keep = [a for a in n.names if (a.asname or a.name) in used]
        else:
            new_body.append(n)
            continue
        removed += len(n.names) - len(keep)
        if keep:
            n.names = keep
            new_body.append(n)
    tree.body = new_body or [ast.Pass()]
    return removed

def make_formats(code, keep_docstrings=False, prune_imports=False):
    """Build B, C, D for one piece of code and verify each by round-trip.
    prune_imports=True also removes unused imports from B/C/D (A is never changed).
    A bad format is returned as None with ok=False and an error string, never silently wrong."""
    res = {"b": None, "c": None, "d": None, "ok_b": False, "ok_c": False, "ok_d": False,
           "err": [], "n_pruned": 0}
    tree = ast.parse(code)
    base = copy.deepcopy(tree)
    if not keep_docstrings:
        strip_docstrings(base)
    if prune_imports:
        res["n_pruned"] = prune_unused_imports(base)
    ref = norm_dump(base)

    b_text, e = _check("B", lambda: ast.unparse(base))
    if b_text is not None:
        ok, e = _check("B", lambda: (norm_dump(b_text) == ref) and (compile(b_text, "<b>", "exec") is not None))
        res["b"], res["ok_b"] = b_text, bool(ok)
        if not ok:
            res["b"] = None
            e = e or "B: round-trip mismatch"
    if e:
        res["err"].append(e)

    for key, mode in (("c", "full"), ("d", "stmt")):
        text, e = _check(key.upper(), lambda m=mode: tree_to_text(base, m))
        if text is not None:
            rebuilt, e = _check(key.upper(), lambda t=text: text_to_source(t))
            if rebuilt is not None:
                ok, e = _check(key.upper(), lambda r=rebuilt: norm_dump(r) == ref)
                if ok:
                    res[key], res["ok_" + key] = text, True
                else:
                    e = e or f"{key.upper()}: round-trip mismatch"
        if e:
            res["err"].append(e)
    return res

print("Converters ready (B canonical code, C compact tree, D statement tree).")

In [ ]:
# CELL 4 — EVALUATION KIT
# Every program under test runs in its OWN Python process with a hard time limit and a memory
# cap, so an infinite loop, sys.exit() or runaway allocation in generated code cannot take down
# the notebook. A test only counts as passed if the runner reaches its final line after the
# tests really ran (earlier notebooks counted a test file that was defined but never run).
import resource  # noqa: F401  (used by the child processes)

PRELUDE = ("import resource\n"
           "try:\n    resource.setrlimit(resource.RLIMIT_AS, (3 << 30, 3 << 30))\nexcept Exception:\n    pass\n")

def run_python(files, entry, args=(), timeout=10):
    """Write {name: text} into a fresh temp dir, run `python entry` there. -> (returncode, stdout, stderr)."""
    with tempfile.TemporaryDirectory() as d:
        for name, text in files.items():
            Path(d, name).write_text((PRELUDE + text) if name == entry else text, encoding="utf-8")
        proc = subprocess.Popen([sys.executable, entry, *map(str, args)], cwd=d, stdin=subprocess.DEVNULL,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, encoding="utf-8",
                                errors="replace", start_new_session=True)
        try:
            out, err = proc.communicate(timeout=timeout)
            return proc.returncode, out, err
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except Exception:
                pass
            try:
                proc.communicate(timeout=5)
            except Exception:
                pass
            return -9, "", "timeout"

HE_RUNNER = '''
import json, sys
spec = json.load(open("spec.json"))
src = open("prog.py", encoding="utf-8").read()
ns = {"__name__": "heval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
    ns["check"](ns[spec["entry"]])
except BaseException as e:
    sys.stderr.write(type(e).__name__ + ": " + str(e)[:300] + "\\n")
    sys.exit(1)
print("__PASS__")
'''

GOLDEN_RUNNER = '''
import json, sys
fname = sys.argv[1]
cases = json.load(open("cases.json"))
ns = {"__name__": "gen_module"}
try:
    exec(compile(open("gen.py", encoding="utf-8").read(), "gen.py", "exec"), ns)
except BaseException as e:
    sys.stderr.write("EXEC " + type(e).__name__ + "\\n"); sys.exit(1)
fn = ns.get(fname)
if not callable(fn):
    sys.stderr.write("NO_FUNCTION\\n"); sys.exit(2)
def norm(x):
    try:
        return json.loads(json.dumps(x))
    except Exception:
        return repr(x)
bad = 0
for c in cases:
    try:
        out = fn(**c["input"])
        ok = ("output" in c) and norm(out) == c["output"]
    except BaseException as e:
        ok = ("exception" in c) and type(e).__name__ == c["exception"]
    bad += (not ok)
if bad:
    sys.stderr.write("FAILED %d/%d\\n" % (bad, len(cases))); sys.exit(3)
print("__PASS__")
'''

CE_RUNNER = '''
import io, json, sys, unittest
src = open("gen.py", encoding="utf-8").read() + "\\n\\n" + open("test.py", encoding="utf-8").read()
ns = {"__name__": "classeval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
except BaseException as e:
    print(json.dumps({"status": "exec_fail", "error": (type(e).__name__ + ": " + str(e))[:200]})); sys.exit(0)
classes = [v for v in ns.values() if isinstance(v, type) and issubclass(v, unittest.TestCase) and v is not unittest.TestCase]
total = failed = 0
for cls in classes:
    res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    total += res.testsRun
    failed += len(res.failures) + len(res.errors)
print(json.dumps({"status": "ok", "tests": total, "failed": failed}))
'''

def run_golden(src, func_name, cases_json, timeout=10):
    rc, out, err = run_python({"runner.py": GOLDEN_RUNNER, "gen.py": src, "cases.json": cases_json},
                              "runner.py", args=[func_name], timeout=timeout)
    return rc == 0 and "__PASS__" in out

def top_level_imports(src):
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return "\n".join(l for l in src.splitlines() if l.startswith(("import ", "from ")))
    keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
    return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""

def humaneval_header(prompt, entry):
    """Imports and fully written helper functions from the prompt (everything except the entry point)."""
    try:
        tree = ast.parse(prompt)
        keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))
                or (isinstance(n, ast.FunctionDef) and n.name != entry)]
        return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""
    except SyntaxError:
        return top_level_imports(prompt)

def run_humaneval(prompt, entry, test, gen_src, timeout=20):   # same limit as when the reference solutions are validated
    prog = humaneval_header(prompt, entry) + "\n\n" + gen_src + "\n\n" + test + "\n"
    rc, out, err = run_python({"runner.py": HE_RUNNER, "prog.py": prog, "spec.json": json.dumps({"entry": entry})},
                              "runner.py", timeout=timeout)
    return (rc == 0 and "__PASS__" in out), (err.strip().splitlines()[-1][:160] if err.strip() else ("timeout" if rc == -9 else ""))

def run_classeval(skeleton, test, gen_src, timeout=30):
    gen = top_level_imports(skeleton) + "\n\n" + gen_src
    rc, out, err = run_python({"runner.py": CE_RUNNER, "gen.py": gen, "test.py": test}, "runner.py", timeout=timeout)
    if rc == -9:
        return {"status": "timeout", "tests": 0, "failed": 0}
    try:
        return json.loads(out.strip().splitlines()[-1])
    except Exception:
        return {"status": "runner_error", "tests": 0, "failed": 0, "error": err.strip()[-160:]}

def defines(src, name, kind):
    try:
        return any(isinstance(n, kind) and n.name == name for n in ast.walk(ast.parse(src)))
    except Exception:
        return False

# ---------------- benchmark data (Hugging Face first, GitHub as a fallback) ----------------
def load_humaneval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("evalplus/humanevalplus", split="test")]
    import gzip, urllib.request
    raw = urllib.request.urlopen("https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz", timeout=60).read()
    return [json.loads(l) for l in gzip.decompress(raw).decode("utf-8").splitlines() if l.strip()]

def load_classeval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("FudanSELab/ClassEval", split="test")]
    import urllib.request
    return json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/FudanSELab/ClassEval/master/data/ClassEval_data.json", timeout=60).read().decode("utf-8"))

HE_REQ = ("task_id", "prompt", "entry_point", "test", "canonical_solution")
CE_REQ = ("task_id", "skeleton", "test", "solution_code", "class_name")
def usable(rows, req):
    """Keep rows that have every field we need as a non-empty string (guards against a differently shaped copy of a benchmark)."""
    return [r for r in rows if all(isinstance(r.get(k), str) and r.get(k) for k in req)]

def canonical_humaneval_src(row):
    tree = ast.parse(row["prompt"] + row["canonical_solution"])
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == row["entry_point"])
    return ast.unparse(node)

def select_humaneval(n, seed):
    """Same selection rule as the earlier runs; keeps only problems whose reference solution passes
    in THIS environment, so a failure can be blamed on the model and not on the setup."""
    best = None
    for source in ("hf", "github"):
        try:
            rows = usable(load_humaneval_rows(source), HE_REQ)
            if not rows:
                raise ValueError("no rows with the expected fields")
        except Exception as e:
            print(f"  HumanEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
            continue
        idx = list(range(len(rows)))
        random.Random(seed).shuffle(idx)
        step = len(idx) / n
        chosen = [rows[i] for i in sorted(idx[int(k * step)] for k in range(n))]
        good = []
        for r in chosen:
            try:
                ok, _ = run_humaneval(r["prompt"], r["entry_point"], r["test"], canonical_humaneval_src(r), timeout=20)
            except Exception:
                ok = False
            if ok:
                good.append(r)
        print(f"  HumanEval from {source}: {len(good)}/{len(chosen)} reference solutions pass the harness")
        if best is None or len(good) > len(best[1]):
            best = (source, good, len(chosen))
        if len(good) >= 0.9 * len(chosen):
            break
    return best

def select_classeval(n, seed, tok):
    for source in ("hf", "github"):
        try:
            rows = usable(load_classeval_rows(source), CE_REQ)
            if not rows:
                raise ValueError("no rows with the expected fields")
            break
        except Exception as e:
            print(f"  ClassEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
    else:
        return None
    fits = [r for r in rows if len(tok(f"<GENERATE> Complete this Python class:\n{r['skeleton']}")["input_ids"]) <= CLASSEVAL_MAX_PROMPT_TOKENS]
    random.Random(seed).shuffle(fits)
    chosen, tried = [], 0
    for r in fits:
        if len(chosen) >= n or tried >= 60:
            break
        tried += 1
        res = run_classeval(r["skeleton"], r["test"], r["solution_code"], timeout=30)
        if res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0:
            chosen.append(r)
    print(f"  ClassEval from {source}: {len(rows)} problems, {len(fits)} fit the encoder window, "
          f"tried {tried}, kept {len(chosen)} whose reference solution passes the harness")
    return source, chosen

print("Evaluation kit ready (isolated processes, hard time limits, tests must actually run).")

## Model utilities, training loop, scoring

In [ ]:
# CELL 5 — TOKENIZER, MODEL, BATCHING, GENERATION, DECODING FORMATS BACK TO PYTHON
from transformers import AutoTokenizer, T5ForConditionalGeneration

def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    tok.model_max_length = 10**7          # we always pass explicit max lengths
    return tok

def load_model():
    return T5ForConditionalGeneration.from_pretrained(BASE_MODEL)

def tokenize_many(tok, texts, chunk=1000):
    out = []
    for i in range(0, len(texts), chunk):
        out.extend(tok(texts[i:i + chunk], truncation=True, max_length=MAX_SRC)["input_ids"])
    return out

def build_examples(frame, fmt, tok):
    ins = tokenize_many(tok, frame["input_text"].tolist())
    labs = tokenize_many(tok, frame[TGT_COL[fmt]].tolist())
    return [{"input_ids": i, "labels": l} for i, l in zip(ins, labs)]

def make_collate(pad_id):
    def collate(batch):
        ms = max(len(b["input_ids"]) for b in batch)
        mt = max(len(b["labels"]) for b in batch)
        ids = torch.full((len(batch), ms), pad_id, dtype=torch.long)
        att = torch.zeros((len(batch), ms), dtype=torch.long)
        lab = torch.full((len(batch), mt), -100, dtype=torch.long)
        for i, b in enumerate(batch):
            n, m = len(b["input_ids"]), len(b["labels"])
            ids[i, :n] = torch.tensor(b["input_ids"]); att[i, :n] = 1
            lab[i, :m] = torch.tensor(b["labels"])
        return {"input_ids": ids, "attention_mask": att, "labels": lab}
    return collate

@torch.no_grad()
def generate_texts(model, tok, texts, max_new_tokens=None, batch_size=None):
    """Greedy, single pass (pass@1). fp32 on purpose: the model is small and this removes a variable."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    batch_size = batch_size or GEN_BATCH
    model.eval()
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))     # similar lengths together
    outs = [None] * len(texts)
    eos = tok.eos_token_id
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC).to(DEVICE)
        seq = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],   # only what T5 uses
                             max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
        for j, i in enumerate(idx):
            row = seq[j].tolist()[1:]                                   # drop the decoder start token
            ended = eos in row
            n = (row.index(eos) + 1) if ended else len(row)
            outs[i] = {"text": tok.decode(seq[j], skip_special_tokens=True, clean_up_tokenization_spaces=False),
                       "n_tokens": n, "ended": ended, "hit_limit": (not ended) and n >= max_new_tokens}
    return outs

def to_source(fmt, text):
    """Model output -> Python source. A and B are already code; C and D go through the deterministic renderer."""
    try:
        if fmt in ("a", "b"):
            return text, None
        return text_to_source(text), None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:100]}"

def ref_source(row, fmt):
    """The reference answer for a row, in the same form the model is asked to produce, rendered to source."""
    if fmt in ("a", "b"):
        return row[TGT_COL[fmt]]
    return text_to_source(row[TGT_COL[fmt]])

def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def fmt_prop(k, n):
    if n == 0:
        return "n/a"
    lo, hi = wilson(k, n)
    return f"{k/n*100:5.1f}% [{lo*100:4.1f}-{hi*100:4.1f}] (n={n})"

def mcnemar_p(b, c):
    """Exact two-sided paired test on the rows where exactly one of two formats succeeded."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)

print("Utilities ready.")

In [ ]:
# CELL 6 — TRAINING LOOP (the same for every format)
def train_format(fmt, tok):
    from torch.utils.data import DataLoader
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    train_ex = build_examples(train_df, fmt, tok)
    val_ex = build_examples(val_df, fmt, tok)
    assert max(len(e["labels"]) for e in train_ex + val_ex) <= MAX_NEW_TOKENS, "a training target exceeds the 512-token budget"
    collate = make_collate(tok.pad_token_id)
    gen = torch.Generator(); gen.manual_seed(SEED)
    train_loader = DataLoader(train_ex, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=gen)
    val_loader = DataLoader(val_ex, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

    model = load_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = steps_per_epoch * EPOCHS
    warmup = max(1, int(total_steps * WARMUP_FRAC))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warmup if s < warmup else max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
    use_amp = DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, best_val, best_state, best_epoch, skipped, t_start = [], float("inf"), None, 0, 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_ep, tot_loss, tot_tok = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            n_tok = int((batch["labels"] != -100).sum())
            if not torch.isfinite(loss):
                skipped += 1
                opt.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / GRAD_ACCUM).backward()
            tot_loss += loss.item() * n_tok; tot_tok += n_tok
            if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
        train_loss = tot_loss / max(1, tot_tok)

        model.eval()
        v_loss, v_tok = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    l = model(**batch).loss
                n_tok = int((batch["labels"] != -100).sum())
                v_loss += l.item() * n_tok; v_tok += n_tok
        val_loss = v_loss / max(1, v_tok)
        if not math.isfinite(val_loss):
            raise RuntimeError(f"validation loss is not finite for format {fmt}; training diverged")
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "seconds": time.time() - t_ep})
        print(f"  [{FORMAT_NAMES[fmt]}] epoch {epoch}/{EPOCHS}  train {train_loss:.4f}  val {val_loss:.4f}  ({time.time()-t_ep:.0f}s)")
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if skipped > 0.05 * len(train_loader) * EPOCHS:
        raise RuntimeError(f"{skipped} batches had a non-finite loss for format {fmt}; training is unstable")
    model.load_state_dict(best_state)
    stats = {"history": history, "best_epoch": best_epoch, "best_val_loss": best_val,
             "train_seconds": time.time() - t_start, "skipped_batches": skipped,
             "n_train": len(train_ex), "n_val": len(val_ex),
             "mean_target_tokens_train": float(np.mean([len(e["labels"]) for e in train_ex])),
             "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 1)}
    return model, stats

print("Training loop ready (fp16 autocast, linear warmup/decay, best-validation-loss weights kept).")

In [ ]:
# CELL 7 — SCORING ONE TRAINED MODEL
# Every output is first turned into Python source (trees go through the deterministic renderer),
# then judged only on that source. Metrics per output:
#   valid        non-empty source that parses as Python
#   compiles     compile() accepts it
#   exact        same AST as the reference answer for that format (docstrings/comments ignored)
#   exact_logic  same, also ignoring unused imports (the model cannot guess a file's import block)
#   sim_logic    partial credit: similarity of node-type/depth counts to the reference (0 to 1)
#   golden       passes the golden input/output cases, in a separate process (only rows that have them)
def parse_tree(src):
    try:
        return ast.parse(src)
    except Exception:
        return None

def canon_tree(tree, keep_doc, prune):
    t = copy.deepcopy(tree)
    if not keep_doc:
        strip_docstrings(t)
    if prune:
        prune_unused_imports(t)
    return t

def node_counts(tree):
    c = Counter()
    def walk(n, d=0):
        c[(type(n).__name__, min(d, 7))] += 1
        for ch in ast.iter_child_nodes(n):
            walk(ch, d + 1)
    walk(tree)
    return c

def struct_sim(t1, t2):
    a, b = node_counts(t1), node_counts(t2)
    keys = set(a) | set(b)
    tot = sum(a.values()) + sum(b.values())
    return 1.0 - sum(abs(a[k] - b[k]) for k in keys) / tot if tot else 1.0

def score_row(row, fmt, gen, golden_ok):
    out = {"valid": False, "compiles": False, "exact": False, "exact_logic": False, "sim_logic": 0.0,
           "golden": (False if golden_ok else None), "render_error": "", "src": ""}   # unusable output = failed, not skipped
    src, err = to_source(fmt, gen["text"])
    out["render_error"] = err or ""
    if src is None or not src.strip():
        return out
    out["src"] = src[:20000]
    tree = parse_tree(src)
    if tree is None:
        return out
    out["valid"] = True
    try:
        compile(tree, "<pred>", "exec"); out["compiles"] = True
    except Exception:
        pass
    keep = bool(row["docstrings_kept"])
    ref_tree = ast.parse(ref_source(row, fmt))
    out["exact"] = norm_dump(canon_tree(tree, keep, False)) == norm_dump(canon_tree(ref_tree, keep, False))
    pt, rt = canon_tree(tree, keep, True), canon_tree(ref_tree, keep, True)
    out["exact_logic"] = norm_dump(pt) == norm_dump(rt)
    out["sim_logic"] = round(struct_sim(pt, rt), 4)
    if golden_ok:
        out["golden"] = bool(run_golden(src, row["golden_func"], row["golden_io_json"]))
    return out

def show_sample(tag, text, src, limit=260):
    print(f"      {tag}: {text[:limit]!r}{' ...' if len(text) > limit else ''}")
    if src is not None and src != text:
        print(f"      -> source: {src[:limit]!r}{' ...' if len(src) > limit else ''}")

def evaluate_format(fmt, model, tok):
    name = FORMAT_NAMES[fmt]
    preds = []

    # ---------- held-out rows of our own dataset ----------
    t0 = time.time()
    gens = generate_texts(model, tok, eval_rows["input_text"].tolist())
    gen_sec = time.time() - t0
    print(f"  [{name}] generated {len(gens)} own-dataset outputs in {gen_sec:.0f}s; scoring...")
    for (_, row), g in zip(eval_rows.iterrows(), gens):
        s = score_row(row, fmt, g, golden_ok=bool(row["golden_ok"]))
        preds.append({"fmt": fmt, "kind": "own", "key": row["id"], "task": row["task"], "unit_type": row["unit_type"],
                      "text": g["text"], "n_tokens": g["n_tokens"], "ended": g["ended"], "hit_limit": g["hit_limit"],
                      "passed": None, "tests_total": None, "tests_failed": None, **s})

    print(f"  [{name}] samples (raw model output, then the Python it renders to):")
    shown = Counter()
    for p in preds:
        if p["kind"] == "own" and shown[p["task"]] < 1:
            shown[p["task"]] += 1
            print(f"    - {p['task']} / {p['unit_type']} / {p['key']}  valid={p['valid']} exact_logic={p['exact_logic']}")
            show_sample("model output", p["text"], p["src"] or None)

    # ---------- HumanEval ----------
    heval = []
    if he_problems:
        gens = generate_texts(model, tok, [f"<GENERATE> Complete this Python function:\n{r['prompt']}" for r in he_problems])
        for r, g in zip(he_problems, gens):
            src, err = to_source(fmt, g["text"])
            tree = parse_tree(src) if (src and src.strip()) else None
            valid = tree is not None
            has_def = valid and defines(src, r["entry_point"], ast.FunctionDef)
            passed = False
            if has_def:
                passed, _ = run_humaneval(r["prompt"], r["entry_point"], r["test"], src)
            preds.append({"fmt": fmt, "kind": "humaneval", "key": r["task_id"], "task": "humaneval", "unit_type": "function",
                          "text": g["text"], "n_tokens": g["n_tokens"], "ended": g["ended"], "hit_limit": g["hit_limit"],
                          "valid": valid, "compiles": valid, "exact": False, "exact_logic": False, "sim_logic": 0.0,
                          "golden": None, "render_error": err or "", "src": (src or "")[:20000],
                          "passed": bool(passed), "tests_total": None, "tests_failed": None, "defines": bool(has_def)})
        k = sum(p["passed"] for p in preds if p["kind"] == "humaneval")
        print(f"  [{name}] HumanEval pass@1: {k}/{len(he_problems)}")

    # ---------- ClassEval ----------
    if ce_problems:
        gens = generate_texts(model, tok, [f"<GENERATE> Complete this Python class:\n{r['skeleton']}" for r in ce_problems])
        for r, g in zip(ce_problems, gens):
            src, err = to_source(fmt, g["text"])
            tree = parse_tree(src) if (src and src.strip()) else None
            valid = tree is not None
            has_cls = valid and defines(src, r["class_name"], ast.ClassDef)
            res = {"status": "not_run", "tests": 0, "failed": 0}
            if has_cls:
                res = run_classeval(r["skeleton"], r["test"], src)
            passed = res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0
            preds.append({"fmt": fmt, "kind": "classeval", "key": r["task_id"], "task": "classeval", "unit_type": "class",
                          "text": g["text"], "n_tokens": g["n_tokens"], "ended": g["ended"], "hit_limit": g["hit_limit"],
                          "valid": valid, "compiles": valid, "exact": False, "exact_logic": False, "sim_logic": 0.0,
                          "golden": None, "render_error": err or "", "src": (src or "")[:20000],
                          "passed": bool(passed), "tests_total": int(res.get("tests", 0)), "tests_failed": int(res.get("failed", 0)),
                          "defines": bool(has_cls)})
        k = sum(p["passed"] for p in preds if p["kind"] == "classeval")
        print(f"  [{name}] ClassEval class-level pass: {k}/{len(ce_problems)}")

    pdf = pd.DataFrame(preds)
    pdf["golden"] = pdf["golden"].astype("boolean")
    pdf["passed"] = pdf["passed"].astype("boolean")
    own = pdf[pdf.kind == "own"]
    gold = own[own.golden.notna()]
    def by(col):
        return {str(k): {"n": int(len(g)), "valid": int(g.valid.sum()), "compiles": int(g.compiles.sum()),
                         "exact_logic": int(g.exact_logic.sum())} for k, g in own.groupby(col)}
    he = pdf[pdf.kind == "humaneval"]; ce = pdf[pdf.kind == "classeval"]
    summary = {
        "own": {"n": int(len(own)), "valid": int(own.valid.sum()), "compiles": int(own.compiles.sum()),
                "exact": int(own.exact.sum()), "exact_logic": int(own.exact_logic.sum()),
                "sim_logic_mean": float(own.sim_logic.mean()), "golden_n": int(len(gold)), "golden_pass": int(gold.golden.sum()),
                "hit_limit": int(own.hit_limit.sum()), "ended": int(own.ended.sum()),
                "mean_gen_tokens": float(own.n_tokens.mean()), "gen_seconds_per_row": gen_sec / max(1, len(own)),
                "by_task": by("task"), "by_unit_type": by("unit_type")},
        "humaneval": {"n": int(len(he)), "valid": int(he.valid.sum()), "defines": int(he["defines"].fillna(False).sum()) if len(he) else 0,
                      "passed": int(he.passed.sum()) if len(he) else 0},
        "classeval": {"n": int(len(ce)), "valid": int(ce.valid.sum()), "defines": int(ce["defines"].fillna(False).sum()) if len(ce) else 0,
                      "passed": int(ce.passed.sum()) if len(ce) else 0,
                      "tests_total": int(ce.tests_total.fillna(0).sum()) if len(ce) else 0,
                      "tests_failed": int(ce.tests_failed.fillna(0).sum()) if len(ce) else 0},
    }
    return summary, pdf

print("Scoring ready.")

## Preflight: checks everything that could fail after hours of training

In [ ]:
# CELL 8 — PREFLIGHT: everything that could fail after hours of training is checked here first
tok = load_tokenizer()
checks = []

# 1) the Parquet's tree formats decode with this notebook's converter, and match the canonical code
sample = common.sample(min(300, len(common)), random_state=0)
bad = 0
for _, r in sample.iterrows():
    try:
        ref = norm_dump(r["tgt_b_canonical"]) if r["ok_b"] else None
        for f in FORMATS:
            if f in ("c", "d"):
                got = norm_dump(text_to_source(r[TGT_COL[f]]))
                if ref is not None and got != ref:
                    bad += 1
    except Exception:
        bad += 1
assert bad == 0, f"{bad} Parquet rows do not decode with the converter; rebuild the Parquet with the dataset notebook"
checks.append(f"tree formats decode and match the canonical code ({len(sample)} rows)")

# 2) tokenizer: text -> ids -> text must be lossless, or a perfect model could still not produce the target
for f in FORMATS:
    texts = common[TGT_COL[f]].tolist()
    ids = tokenize_many(tok, texts)
    back = [tok.decode(i, skip_special_tokens=True, clean_up_tokenization_spaces=False) for i in ids]
    mism = [k for k, (a, b) in enumerate(zip(texts, back)) if a != b]
    msg = f"tokenizer round trip, {FORMAT_NAMES[f]}: {len(texts)-len(mism)}/{len(texts)} identical"
    if mism:
        a, b = texts[mism[0]], back[mism[0]]
        d = next((k for k, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
        msg += f"   WARNING: first difference near {a[max(0,d-30):d+30]!r} vs {b[max(0,d-30):d+30]!r}"
    checks.append(msg)
    n = min(500, len(texts))
    fresh = [len(x) for x in ids[:n]]
    col = common[f"tok_{f}"].tolist()[:n]
    off = sum(1 for x, y in zip(fresh, col) if x != y)
    if off:
        checks.append(f"WARNING: {off}/{n} token counts differ from the Parquet's {f'tok_{f}'} column (built with a different tokenizer version?)")

# 3) worst-case memory: a full batch of 512-token inputs and 512-token targets, forward + backward
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    m = load_model().to(DEVICE); m.train()
    o = torch.optim.AdamW(m.parameters(), lr=1e-6)
    v = min(1000, m.config.vocab_size)
    ids = torch.randint(5, v, (BATCH_SIZE, MAX_SRC), device=DEVICE)
    lab = torch.randint(5, v, (BATCH_SIZE, MAX_NEW_TOKENS), device=DEVICE)
    with torch.amp.autocast("cuda", dtype=torch.float16):
        l = m(input_ids=ids, attention_mask=torch.ones_like(ids), labels=lab).loss
    l.backward(); o.step()
    checks.append(f"worst-case batch ({BATCH_SIZE} x 512 in / 512 out) fits: peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB of {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")
    del m, o, l, ids, lab
    gc.collect(); torch.cuda.empty_cache()

# 4) smoke test of the real training + generation code on a handful of rows
def smoke_train(fmt):
    global train_df, val_df, EPOCHS
    saved = (train_df, val_df, EPOCHS)
    train_df, val_df, EPOCHS = saved[0].head(BATCH_SIZE * GRAD_ACCUM * 2), saved[1].head(BATCH_SIZE), 1
    try:
        model, st = train_format(fmt, tok)
    finally:
        train_df, val_df, EPOCHS = saved
    outs = generate_texts(model, tok, saved[1]["input_text"].head(2).tolist(), max_new_tokens=24)
    assert len(outs) == 2 and all("text" in o for o in outs)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
smoke_train(FORMATS[-1])
checks.append(f"training + generation code runs end to end (smoke test on format {FORMATS[-1].upper()})")

# 5) benchmark problems; only ones whose reference solution passes the harness here are kept
print("Selecting benchmark problems (the reference solutions are run through the harness)...")
try:
    he_sel = select_humaneval(HE_N, SEED)
except Exception as e:
    print(f"  HumanEval selection failed ({type(e).__name__}: {str(e)[:100]}); it will be skipped")
    he_sel = None
he_problems = he_sel[1] if he_sel else []
checks.append(f"HumanEval: {len(he_problems)} problems usable" + ("" if he_problems else "   WARNING: none loaded; HumanEval will be skipped"))
try:
    ce_sel = select_classeval(CLASSEVAL_N, SEED, tok)
except Exception as e:
    print(f"  ClassEval selection failed ({type(e).__name__}: {str(e)[:100]}); it will be skipped")
    ce_sel = None
ce_problems = ce_sel[1] if ce_sel else []
checks.append(f"ClassEval: {len(ce_problems)} problems usable" + ("" if ce_problems else "   WARNING: none loaded; ClassEval will be skipped"))

# 6) rows of our own dataset used for generation; golden cases are trusted only if the reference code passes them
gold_rows = test_df[test_df.has_golden_io.astype(bool)]
golden_valid_ids = {r["id"] for _, r in gold_rows.iterrows()
                    if run_golden(r["tgt_a_code"], r["golden_func"], r["golden_io_json"])}
rest = test_df[~test_df["id"].isin(set(gold_rows["id"]))]
n_fill = max(0, EVAL_OWN_N - len(gold_rows))
eval_rows = pd.concat([gold_rows, rest.sample(min(n_fill, len(rest)), random_state=SEED)]).reset_index(drop=True)
eval_rows["golden_ok"] = eval_rows["id"].isin(golden_valid_ids)
checks.append(f"own-dataset evaluation rows: {len(eval_rows)} (golden I/O usable on {int(eval_rows.golden_ok.sum())} of {len(gold_rows)} rows that have it)")
checks.append("tasks in the evaluation rows: " + str(eval_rows.task.value_counts().to_dict()) + " | unit types: " + str(eval_rows.unit_type.value_counts().to_dict()))

print("\nPREFLIGHT")
for c in checks:
    print("  " + ("!! " if "WARNING" in c else "ok ") + c)

## Train and evaluate each format

In [ ]:
# CELL 9 — TRAIN AND EVALUATE EVERY FORMAT (same data, same 7 epochs, same settings)
# Each finished format is saved right away. If Colab disconnects, re-run Cells 1-8 and this cell:
# formats that already finished are skipped.
state = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}
state.setdefault("formats", {})
state["config"] = {
    "formats": FORMATS, "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
    "train_fraction": TRAIN_FRACTION, "max_new_tokens": MAX_NEW_TOKENS, "base_model": BASE_MODEL, "seed": SEED,
    "imports_pruned": bool(df.imports_pruned.any()), "n_train": int(len(train_df)), "n_val": int(len(val_df)),
    "n_test_pool": int(len(test_df)), "n_eval_own": int(len(eval_rows)), "n_humaneval": len(he_problems),
    "n_classeval": len(ce_problems), "torch": torch.__version__, "transformers": transformers.__version__,
    "python": sys.version.split()[0],
}
if "baseline_copy" not in state:
    # Reference point: "answer" every repair/refactor row by copying the input code unchanged.
    sc = []
    for _, r in eval_rows[eval_rows.task.isin(["repair", "refactor"])].iterrows():
        if isinstance(r["tgt_b_canonical"], str):
            sc.append(score_row(r, "b", {"text": r["context_code"]}, bool(r["golden_ok"])))
    gl = [s["golden"] for s in sc if s["golden"] is not None]
    state["baseline_copy"] = {"n": len(sc), "valid": int(sum(s["valid"] for s in sc)), "compiles": int(sum(s["compiles"] for s in sc)),
                              "exact_logic": int(sum(s["exact_logic"] for s in sc)), "golden_n": len(gl), "golden_pass": int(sum(gl))}
RESULTS_FILE.write_text(json.dumps(state, indent=1))

for fmt in FORMATS:
    pred_file = WORK / f"preds_{fmt}.parquet"
    if fmt in state["formats"] and pred_file.exists():
        print(f"{FORMAT_NAMES[fmt]}: already finished, skipping")
        continue
    print("=" * 90)
    print(f"FORMAT {FORMAT_NAMES[fmt]}: training {EPOCHS} epochs on {len(train_df)} rows")
    print("=" * 90)
    model, stats = train_format(fmt, tok)
    model.save_pretrained(WORK / fmt / "best")
    print(f"  trained in {stats['train_seconds']/60:.1f} min; best epoch {stats['best_epoch']} (val loss {stats['best_val_loss']:.4f})")
    summary, pdf = evaluate_format(fmt, model, tok)
    pdf.to_parquet(pred_file)
    state["formats"][fmt] = {"train": stats, "eval": summary}
    RESULTS_FILE.write_text(json.dumps(state, indent=1))
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"  saved {pred_file.name} and results.json")

print("\nAll formats finished. Run the next cell for the comparison and conclusions.")

## Comparison and conclusions

In [ ]:
# CELL 10 — COMPARISON AND CONCLUSIONS (every statement below is computed from the results)
state = json.loads(RESULTS_FILE.read_text())
fmts = [f for f in FORMATS if f in state["formats"]]
assert fmts, "no finished format found; run Cell 9 first"
preds = {f: pd.read_parquet(WORK / f"preds_{f}.parquet") for f in fmts}
cfg = state["config"]
lines = []
def out(s=""):
    lines.append(s); print(s)
def nm(f):
    return FORMAT_NAMES[f]

out("=" * 104)
out(f"FORMAT EXPERIMENT: CodeT5-small, {cfg['epochs']} epochs per format, same rows, same settings")
out("=" * 104)
out(f"training rows {cfg['n_train']} | validation rows {cfg['n_val']} | evaluation rows {cfg['n_eval_own']} (held-out code units) | "
    f"HumanEval problems {cfg['n_humaneval']} | ClassEval problems {cfg['n_classeval']}")
out(f"learning rate {cfg['lr']}, batch {cfg['batch_size']}x{cfg['grad_accum']}, greedy decoding up to {cfg['max_new_tokens']} tokens, imports pruned in B/C/D: {cfg['imports_pruned']}")

# ---------------------------------------------------------------- 1. training
out("\n1) TRAINING")
out(f"   {'format':12s} {'target tokens':>13s} {'best epoch':>10s} {'best val loss':>13s} {'minutes':>8s}   validation loss by epoch")
for f in fmts:
    t = state["formats"][f]["train"]
    curve = " ".join(f"{h['val_loss']:.2f}" for h in t["history"])
    out(f"   {nm(f):12s} {t['mean_target_tokens_train']:13.0f} {t['best_epoch']:10d} {t['best_val_loss']:13.4f} {t['train_seconds']/60:8.1f}   {curve}")
out("   Validation loss is NOT comparable across formats (different token counts and vocabulary use); use the generation results below.")
for f in fmts:
    if state["formats"][f]["train"]["skipped_batches"]:
        out(f"   note: {nm(f)} skipped {state['formats'][f]['train']['skipped_batches']} batches with a non-finite loss")

# ---------------------------------------------------------------- 2. own data
out("\n2) GENERATION ON HELD-OUT ROWS OF OUR DATASET (each output is turned into Python first; 95% intervals in brackets)")
def block(title, getter, ref=None):
    out(f"   {title}")
    for f in fmts:
        k, n = getter(state["formats"][f]["eval"]["own"])
        out(f"      {nm(f):12s} {fmt_prop(k, n)}")
    if ref:
        out(f"      {'(reference)':12s} {ref}")
bc = state.get("baseline_copy")
def bref(key, nkey="n"):
    return (f"copy the input unchanged, repair+refactor rows only: {fmt_prop(bc[key], bc[nkey])}" if bc and bc[nkey] else None)
block("valid Python", lambda o: (o["valid"], o["n"]), bref("valid"))
block("exact match with the reference answer (docstrings/comments ignored)", lambda o: (o["exact"], o["n"]))
block("exact match, also ignoring unused imports", lambda o: (o["exact_logic"], o["n"]), bref("exact_logic"))
block("passes golden input/output cases (only typed top-level functions have them)", lambda o: (o["golden_pass"], o["golden_n"]),
      (f"copy the input unchanged: {fmt_prop(bc['golden_pass'], bc['golden_n'])}" if bc and bc["golden_n"] else None))
block("ran to the token limit without finishing (looping or cut off)", lambda o: (o["hit_limit"], o["n"]))
out(f"   mean structural similarity to the reference (0-1): " + ", ".join(f"{nm(f)} {state['formats'][f]['eval']['own']['sim_logic_mean']:.3f}" for f in fmts))
out("   Each format is judged against its own reference answer; 'ignoring unused imports' removes the part of the answer no model can know.")
out("   A refactor must keep behaviour, so copying the input passes golden I/O there; read golden results next to that reference.")

out("\n   by task: valid / exact match ignoring unused imports")
for task in ("generate", "repair", "refactor"):
    cells = []
    for f in fmts:
        t = state["formats"][f]["eval"]["own"]["by_task"].get(task)
        cells.append(f"{nm(f)} {t['valid']}/{t['n']} | {t['exact_logic']}/{t['n']}" if t else f"{nm(f)} n/a")
    out(f"      {task:9s} " + "    ".join(cells))
out("   by unit type: valid / exact match ignoring unused imports")
for ut in sorted({u for f in fmts for u in state["formats"][f]["eval"]["own"]["by_unit_type"]}):
    cells = []
    for f in fmts:
        t = state["formats"][f]["eval"]["own"]["by_unit_type"].get(ut)
        cells.append(f"{nm(f)} {t['valid']}/{t['n']} | {t['exact_logic']}/{t['n']}" if t else f"{nm(f)} n/a")
    out(f"      {ut:15s} " + "    ".join(cells))

# ---------------------------------------------------------------- 3. benchmarks
out("\n3) BENCHMARKS (zero-shot, single greedy pass; only problems whose reference solution passes the harness are used)")
for f in fmts:
    he, ce = state["formats"][f]["eval"]["humaneval"], state["formats"][f]["eval"]["classeval"]
    out(f"   {nm(f):12s} HumanEval pass {he['passed']}/{he['n']} (valid Python {he['valid']}, defines the function {he['defines']})"
        f"   |   ClassEval class-level pass {ce['passed']}/{ce['n']}, tests passed {ce['tests_total']-ce['tests_failed']}/{ce['tests_total']} "
        f"(valid {ce['valid']}, defines the class {ce['defines']})")

# ---------------------------------------------------------------- 4. paired comparisons
out("\n4) PAIRED COMPARISONS (same rows for both formats; exact paired test; p < 0.05 counts as a difference)")
METRICS = [("own", "valid", "valid Python"), ("own", "exact_logic", "exact match (ignoring unused imports)"),
           ("own", "golden", "golden I/O"), ("humaneval", "passed", "HumanEval"), ("classeval", "passed", "ClassEval")]
def paired(kind, col, f1, f2):
    a = preds[f1][preds[f1].kind == kind].set_index("key")[col]
    b = preds[f2][preds[f2].kind == kind].set_index("key")[col]
    idx = a.index.intersection(b.index)
    a, b = a.loc[idx], b.loc[idx]
    keep = a.notna() & b.notna()
    a, b = a[keep].astype(bool), b[keep].astype(bool)
    return int(a.sum()), int(b.sum()), len(a), int((a & ~b).sum()), int((~a & b).sum())
pairs = [(fmts[i], fmts[j]) for i in range(len(fmts)) for j in range(i + 1, len(fmts))]
differences = []
n_tests = 0
for kind, col, label in METRICS:
    out(f"   {label}")
    for f1, f2 in pairs:
        k1, k2, n, b01, c01 = paired(kind, col, f1, f2)
        if n < 5 or (k1 == 0 and k2 == 0):
            verdict = "not measurable here (too few rows or both zero)"
        else:
            p = mcnemar_p(b01, c01)
            n_tests += 1
            if p < 0.05:
                hi, lo = (f1, f2) if k1 > k2 else (f2, f1)
                verdict = f"{nm(hi)} higher (p={p:.3f})"
                differences.append((label, hi, lo, p))
            else:
                verdict = f"no clear difference (p={p:.2f})"
        out(f"      {nm(f1)} {k1}/{n}  vs  {nm(f2)} {k2}/{n}   ->  {verdict}")

# ---------------------------------------------------------------- 5. conclusions
out("\n" + "=" * 104)
out("CONCLUSIONS (generated from the numbers above)")
out("=" * 104)
own = {f: state["formats"][f]["eval"]["own"] for f in fmts}
problems = False
for f in fmts:
    o = own[f]
    if o["valid"] / o["n"] < 0.10:
        problems = True
        out(f" - {nm(f)} produced valid Python for only {o['valid']}/{o['n']} outputs, so its other numbers mostly reflect that.")
    if o["hit_limit"] / o["n"] > 0.30:
        problems = True
        out(f" - {nm(f)}: {o['hit_limit']}/{o['n']} outputs ran to the {cfg['max_new_tokens']}-token limit without finishing (looping or cut off).")
if all(own[f]["exact_logic"] == 0 and own[f]["golden_pass"] == 0 for f in fmts):
    out(" - NO FORMAT reproduced a single reference answer or passed a golden case. These runs cannot rank the formats;")
    out("   look at the printed samples, the validation losses and the 'valid Python' rates instead.")
elif differences:
    out(" - Differences larger than chance (paired test, uncorrected for the number of comparisons):")
    for label, hi, lo, p in differences:
        out(f"     * {label}: {nm(hi)} is higher than {nm(lo)} (p={p:.3f})")
else:
    out(" - No metric shows a difference between formats larger than chance, on the rows used here.")
if len(fmts) > 1:
    base = fmts[0]
    for f in fmts[1:]:
        out(f" - {nm(f)} needs {state['formats'][f]['train']['mean_target_tokens_train']/state['formats'][base]['train']['mean_target_tokens_train']:.2f}x the output tokens of "
            f"{nm(base)} on the same rows; generation took {own[f]['gen_seconds_per_row']/max(1e-9, own[base]['gen_seconds_per_row']):.2f}x as long per row "
            f"({own[f]['gen_seconds_per_row']:.2f}s vs {own[base]['gen_seconds_per_row']:.2f}s).")
for name, key in (("HumanEval", "humaneval"), ("ClassEval", "classeval")):
    tot = {f: state["formats"][f]["eval"][key]["passed"] for f in fmts}
    n = state["formats"][fmts[0]]["eval"][key]["n"]
    if n == 0:
        out(f" - {name} was not run (no problems could be loaded).")
    elif all(v == 0 for v in tot.values()):
        out(f" - {name}: every format scored 0/{n}. At this model size and training budget it cannot separate the formats.")
    else:
        out(f" - {name}: " + ", ".join(f"{nm(f)} {v}/{n}" for f, v in tot.items()) + f" (n={n}; one problem is worth {100/n:.0f} points, so small gaps are noise).")

out("\nWhat these results can and cannot say:")
n_own = cfg["n_eval_own"]; gold_n = max(own[f]["golden_n"] for f in fmts)
out(f" - One run per format: one seed, one learning rate ({cfg['lr']}), {cfg['epochs']} epochs. A format that needs a different setting is not given one.")
out(f" - {n_own} evaluation rows, of which {gold_n} have golden I/O; HumanEval has {cfg['n_humaneval']} problems and ClassEval {cfg['n_classeval']}.")
out(" - Only rows that fit in 512 tokens in every format were used, so long code is under-represented; async code and refactors are the most affected.")
out(" - ClassEval problems were chosen among those whose skeleton fits the encoder window, so they are the short ones.")
out(f" - {n_tests} paired tests were run at p<0.05, so about {n_tests*0.05:.1f} apparent differences would appear by chance alone.")
out(" - The model reads raw code as input in every format; only the output format changes.")
Path(WORK / "report.txt").write_text("\n".join(lines), encoding="utf-8")
print(f"\n(report also saved to {WORK / 'report.txt'})")

In [ ]:
# CELL 11 — DOWNLOAD: results, report, and every prediction (to inspect failures later)
from google.colab import files
parts = [pd.read_parquet(WORK / f"preds_{f}.parquet") for f in FORMATS if (WORK / f"preds_{f}.parquet").exists()]
pd.concat(parts, ignore_index=True).to_parquet(WORK / "predictions_all.parquet")
for name in ("results.json", "report.txt", "predictions_all.parquet"):
    if (WORK / name).exists():
        print("downloading", name, f"({(WORK / name).stat().st_size/1e6:.1f} MB)")
        files.download(str(WORK / name))